# LoRA distillation: teach the local 0.5B student (Colab T4)

Train Qwen/Qwen2.5-0.5B-Instruct with a LoRA adapter on the
teacher-verified dataset, and evaluate after every epoch on the
held-out set. Nothing here runs on the VM: this notebook runs on
a free Colab T4 GPU.

Plan: `docs/DISTILLATION_PLAN.md` section 3. Dataset:
`data/distillation/train.jsonl` and `heldout.jsonl` in Google Drive
(see the paths cell). Teacher: gemini/gemini-3.5-flash-lite,
18 Nov 2026. Base model revision 7ae55760, same as production.

Do NOT run this notebook twice without copying the adapter output
first: the output folder is overwritten on each run.

## 0. Runtime check

Use Runtime > Change runtime type > T4 GPU before running anything.
This cell fails loudly on a CPU runtime.

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv"], capture_output=True, text=True)
print(out.stdout)
assert "T4" in out.stdout, "not a T4 GPU runtime, stop here"
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
assert torch.cuda.is_available()

## 1. Mount Drive and set paths

Upload the `data/distillation/` folder to Drive first, at
`MyDrive/rag-distillation/data/distillation/`. The adapter and the
per-epoch results are written to
`MyDrive/rag-distillation/adapters/qwen2.5-0.5b-distilled-v1/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DATA_DIR = Path('/content/drive/MyDrive/rag-distillation/data/distillation')
ADAPTER_DIR = Path('/content/drive/MyDrive/rag-distillation/adapters/qwen2.5-0.5b-distilled-v1')
CHECKPOINT_DIR = Path('/content/drive/MyDrive/rag-distillation/checkpoints')
for name in ('train.jsonl', 'heldout.jsonl'):
    assert (DATA_DIR / name).exists(), f'missing {DATA_DIR / name}, upload the dataset first'
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print('dataset:', DATA_DIR)
print('adapter output:', ADAPTER_DIR)

## 2. Install training packages and clone the repo

Training packages only. They stay out of the repo `requirements.txt`.
The repo clone gives us the production verification code, so the
held-out eval uses the exact same `verify_answer` as deployment.

In [ ]:
%pip install -q peft trl datasets accelerate rank_bm25
%pip uninstall -y torchao  # peft 0.21.2 needs torchao >0.16, Colab ships 0.10; torchao is unused here
import peft, trl, datasets, accelerate, transformers
print('peft', peft.__version__)
print('trl', trl.__version__)
print('datasets', datasets.__version__)
print('accelerate', accelerate.__version__)
print('transformers', transformers.__version__)

import subprocess
subprocess.run(['git', 'clone', '-q',
                'https://github.com/waseem1302-x/production-rag-eval-harness.git',
                '/content/production-rag-eval-harness'], check=True)
import sys
sys.path.insert(0, '/content/production-rag-eval-harness')
from app.verification import verify_answer
print('production verification imported')


## 3. Load the dataset and measure prompt lengths

The training prompt is the exact deployed prompt (the `user`
message in each record). The completion is the verified teacher
answer. Prompt tokens are masked, so the loss is on the completion
only. If the p95 prompt length exceeds 1024 tokens, the max length
rises to 1280. Never truncate the context.

In [ ]:
import json
import numpy as np
from transformers import AutoTokenizer

BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def load_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]

train_records = load_jsonl(DATA_DIR / 'train.jsonl')
heldout_records = load_jsonl(DATA_DIR / 'heldout.jsonl')
print('train records:', len(train_records))
print('heldout records:', len(heldout_records))

def prompt_text(record):
    user = record['messages'][0]['content']
    return tokenizer.apply_chat_template(
        [{'role': 'user', 'content': user}], tokenize=False,
        add_generation_prompt=True)

def completion_text(record):
    return record['messages'][1]['content'] + tokenizer.eos_token

lengths = [len(tokenizer(prompt_text(r))['input_ids']) for r in train_records]
p95 = int(np.percentile(lengths, 95))
MAX_LEN = 1280 if p95 > 1024 else 1024
print(f'prompt tokens: p50={int(np.percentile(lengths, 50))} '
      f'p95={p95} max={max(lengths)} -> MAX_LEN={MAX_LEN}')

## 4. Held-out eval (base model first)

Run the base model on the held-out set BEFORE training, so the
adapter comparison is real. A record counts as correct when the
generated answer contains the expected phrase and the production
verifier keeps at least one sentence. Judge records are scored
separately: the two lines must parse and equal the expected bits.

In [ ]:
import json, torch
from app.textnorm import norm_text

chunks = [json.loads(line) for line in
          Path('/content/production-rag-eval-harness/data/processed/chunks.jsonl')
          .read_text().splitlines() if line.strip()]
CHUNK_TEXT = {c['chunk_id']: c for c in chunks}

def eval_heldout(model, tokenizer, records, batch_size=8):
    model.eval()
    tokenizer.padding_side = 'left'
    gen_correct = gen_total = 0
    judge_correct = judge_total = 0
    per_record = []
    with torch.no_grad():
        for i in range(0, len(records), batch_size):
            batch = records[i:i + batch_size]
            prompts = [prompt_text(r) for r in batch]
            inputs = tokenizer(prompts, return_tensors='pt', padding=True).to(model.device)
            out = model.generate(**inputs, max_new_tokens=120, do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id)
            for j, record in enumerate(batch):
                gen = out[j][inputs['input_ids'].shape[1]:]
                text = tokenizer.decode(gen, skip_special_tokens=True).strip()
                if record['source'] == 'judge':
                    import re
                    fm = re.search(r'faithfulness\s*:\s*([01])', text, re.IGNORECASE)
                    rm = re.search(r'relevance\s*:\s*([01])', text, re.IGNORECASE)
                    exp = record['expected_judge']
                    ok = (fm is not None and rm is not None
                          and int(fm.group(1)) == exp['faithfulness']
                          and int(rm.group(1)) == exp['relevance'])
                    judge_total += 1
                    judge_correct += ok
                    per_record.append({'id': record['question_id'], 'kind': 'judge',
                                       'correct': ok, 'output': text})
                    continue
                expected = record.get('expected_phrase') or ''
                phrase_ok = norm_text(expected.lower()) in norm_text(text.lower())
                ev_chunks = [CHUNK_TEXT[cid] for cid in record['evidence_chunks']]
                kept = verify_answer(text, ev_chunks)
                ok = phrase_ok and bool(kept['kept'])
                gen_total += 1
                gen_correct += ok
                per_record.append({'id': record['question_id'], 'kind': 'generation',
                                   'correct': ok, 'output': text})
    model.train()
    tokenizer.padding_side = 'right'
    return {'generation': {'correct': gen_correct, 'total': gen_total,
                           'accuracy': round(gen_correct / max(1, gen_total), 4)},
            'judge': {'correct': judge_correct, 'total': judge_total,
                      'accuracy': round(judge_correct / max(1, judge_total), 4)},
            'per_record': per_record}

## 5. Load the base model and the LoRA config

LoRA rank 16, alpha 32, dropout 0.05, all linear layers. About 9M
trainable params (1.8 percent of 0.5B). bf16, same as production.

In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float16,
                                             attn_implementation='sdpa').cuda()
model.config.use_cache = False
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'])
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

BASELINE = eval_heldout(model, tokenizer, heldout_records)
print('BASELINE held-out:', json.dumps(BASELINE['generation']), json.dumps(BASELINE['judge']))
(ADAPTER_DIR / 'baseline_heldout.json').write_text(json.dumps(BASELINE, indent=2))


## 6. Train, with eval after every epoch

3 epochs maximum, effective batch 16, lr 2e-4 cosine with 5 percent
warmup. Checkpoints save to Drive after every epoch. The callback
runs the held-out eval at each epoch end and saves the best adapter
to Drive. Keep the best epoch, not the last. If epoch 1 is already
worse than the baseline, stop: do not train longer hoping it recovers.

In [ ]:
from datasets import Dataset
from transformers import TrainerCallback
from trl import SFTConfig, SFTTrainer

train_data = Dataset.from_list([
    {'prompt': prompt_text(r), 'completion': completion_text(r)}
    for r in train_records])

CURRENT = {}
EPOCH_RESULTS = []

class HeldOutEvalCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        result = eval_heldout(CURRENT['trainer'].model, tokenizer, heldout_records)
        epoch = round(state.epoch, 2)
        entry = {'epoch': epoch, 'generation': result['generation'],
                 'judge': result['judge']}
        EPOCH_RESULTS.append(entry)
        (ADAPTER_DIR / 'epoch_results.json').write_text(json.dumps(EPOCH_RESULTS, indent=2))
        print(f'epoch {epoch}: held-out generation {result["generation"]} judge {result["judge"]}', flush=True)
        best = max(EPOCH_RESULTS, key=lambda e: (e['generation']['accuracy'], e['judge']['accuracy']))
        if entry is best:
            CURRENT['trainer'].model.save_pretrained(str(ADAPTER_DIR))
            print(f'epoch {epoch}: new best, adapter saved to Drive', flush=True)
        if (entry['generation']['accuracy'] < BASELINE['generation']['accuracy']
                and epoch >= 1.0):
            print('epoch worse than baseline, stopping early', flush=True)
            control.should_training_stop = True

args = SFTConfig(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=3,
    fp16=True,
    gradient_checkpointing=True,
    optim='adamw_torch_fused',
    logging_steps=5,
    save_strategy='epoch',
    save_total_limit=3,
    max_length=MAX_LEN,
    completion_only_loss=True,
    packing=False,
    report_to=[],
)
trainer = SFTTrainer(model=model, args=args, train_dataset=train_data,
                     processing_class=tokenizer,
                     callbacks=[HeldOutEvalCallback()])
CURRENT['trainer'] = trainer
trainer.train()
print('training done')


## 7. Save the adapter and the training record

The best epoch adapter is already in `ADAPTER_DIR` (the callback
saved it). This cell writes `training.json` with the teacher, the
LoRA config, the epoch kept, and the held-out scores, then lists
the files. Copy the whole adapter folder off Drive before the next
run overwrites it.

In [ ]:
training_record = {
    'base_model': BASE_MODEL,
    'base_revision': '7ae55760',
    'teacher': 'gemini/gemini-3.5-flash-lite',
    'teacher_date': '2026-11-18',
    'train_records': len(train_records),
    'heldout_records': len(heldout_records),
    'lora': {'r': 16, 'alpha': 32, 'dropout': 0.05,
             'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                                'gate_proj', 'up_proj', 'down_proj']},
    'learning_rate': 2e-4, 'epochs_max': 3, 'effective_batch': 16,
    'max_length': MAX_LEN,
    'baseline_heldout': {'generation': BASELINE['generation'],
                         'judge': BASELINE['judge']},
    'epoch_results': EPOCH_RESULTS,
}
(ADAPTER_DIR / 'training.json').write_text(json.dumps(training_record, indent=2))
print(json.dumps(training_record, indent=2))
print('adapter files:', sorted(p.name for p in ADAPTER_DIR.iterdir()))

## Next steps (on the VM, not here)

1. Copy the adapter folder from Drive into the repo at
   `adapters/qwen2.5-0.5b-distilled-v1/`.
2. Run the 68-question generative eval on the VM with the adapter
   (`--adapter` flag, see the plan section 5) and the base model on
   the held-out set for the comparison.
3. Decide by the keep rule in the plan section 4. If it ships,
   integrate, test, update the docs, and commit on a feat branch.
   If it fails, record the negative result in EVAL_RESULTS.md and
   keep the base model.